# The Task

## Project Overview

In the file dataset/data.csv, you will find a dataset containing news articles with the following columns:

label: 0 if the news is fake, 1 if the news is real.
title: The headline of the news article.
text: The full content of the article.
subject: The category or topic of the news.
date: The publication date of the article.
Your goal is to build a classifier that is able to distinguish between the two.

Once you have a classifier built, then use it to predict the labels for dataset/validation_data.csv. Generate a new file where the label 2 has been replaced by 0 (fake) or 1 (real) according to your model. Please respect the original file format, do not include extra columns, and respect the column separator.

Please ensure to split the data.csv into training and test datasets before using it for model training or evaluation.

Guidance
Like in a real life scenario, you are able to make your own choices and text treatment. Use the techniques you have learned and the common packages to process this data and classify the text.

Deliverables
Python Code: Provide well-documented Python code that conducts the analysis.
Predictions: A csv file in the same format as validation_data.csv but with the predicted labels (0 or 1)
Accuracy estimation: Provide the teacher with your estimation of how your model will perform.
Presentation: You will present your model in a 10-minute presentation. Your teacher will provide further instructions.

# Import

In [ ]:
import re
import nltk
import string
import numpy as np
import pandas as pd
import seaborn as sb
from nltk import pos_tag
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.naive_bayes import MultinomialNB
from nltk.stem.snowball import SnowballStemmer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, auc, roc_curve

# Loading the Data

In [ ]:
data = pd.read_csv('../dataset/training_data_lowercase.csv', sep='\t', names=['labels', 'text'])

In [ ]:
data

# Understanding the Data

In [ ]:
data.info()

In [ ]:
print('Its unique values are ',data['labels'].unique())
print(print(data['labels'].describe()))

In [ ]:
plt.hist(data.labels, color='red')
plt.show()

In [ ]:
data.head()

In [ ]:
data.tail()

# Preprocessing the Data

## Basic cleaning 

In [ ]:
def clean_html_text(text: str) -> str:
    if text is None:
        return ''
    text = str(text)
    # Remove inline JavaScript/CSS
    text = re.sub(r"(?is)<script.*?>.*?</script>", " ", text)
    text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
    # Remove HTML comments
    text = re.sub(r"(?s)<!--.*?-->", " ", text)
    # Remove the remaining tag
    text = re.sub(r"(?s)<[^>]+>", " ", text)
    # Remove prefixed b
    text = re.sub(r"^\s*b[\"'](.+?)[\"']\s*$", r"\1", text)
    # Remove video
    # text = re.sub(r"\s*\[video\]$", r"\1", text)    
    # Remove end of the line characters
    text = re.sub(r"\s*[\[\(][^\]\)]+[\]\)]\s*$", "", text)    
    # Remove \t from middle and end of the texts
    text = re.sub(r"\b\\t"," ",text)
    # Remove \t from startof the texts
    text = re.sub(r"^\\t"," ",text)
    # Remove all the special characters and numbers
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    # Remove all single characters
    text = re.sub(r"\b[A-Za-z]\b", " ", text)
    # Remove single characters from the start
    text = re.sub(r"^[A-Za-z]\s+", " ", text)
    # Substitute multiple spaces with single space
    text = re.sub(r"\s+", " ", text).strip()
    # Convert to lowercase
    text = text.lower()
    return text

punct_pattern = f"[{re.escape(string.punctuation)}]"

In [ ]:
data['pre_text'] = data['text'].astype(str).apply(lambda x: clean_html_text(x))
data['pre_text'] = data['pre_text'].astype(str).apply(lambda x: re.sub(punct_pattern, "", x))
data['pre_text'] = data['pre_text'].astype(str).apply(lambda x: word_tokenize(x))
data.head()

## Removing stop words

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
data['pre_text'] = data['pre_text'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

In [ ]:
bag_of_words = {}

for lista in data['pre_text']:
    for word in lista:
        if bag_of_words == 0:
            bag_of_words[word] = 1
        elif word in bag_of_words:
            bag_of_words[word] +=1
        else:
            bag_of_words[word] = 1

print(sorted(bag_of_words.items(), key=lambda x: -x[1])[:100])

In [ ]:
words_to_filter = ['video','says', 'tweets', 'tells','screenshots',
                   'details', 'fck', 'btch', 'images', 'cck', 'image'
                   ,'videos','ahole']

In [ ]:
data['pre_text_filter'] = data['pre_text'].apply(lambda tokens: [word for word in tokens if word not in words_to_filter])

In [ ]:
bag_of_words = {}

for lista in data['pre_text_filter']:
    for word in lista:
        if bag_of_words == 0:
            bag_of_words[word] = 1
        elif word in bag_of_words:
            bag_of_words[word] +=1
        else:
            bag_of_words[word] = 1

print(sorted(bag_of_words.items(), key=lambda x: -x[1])[:100])

### Using Stemmer

#### Snowball

In [ ]:
snowball = SnowballStemmer('english')

In [ ]:
data['snow_text'] = data['pre_text'].apply(lambda tokens: [snowball.stem(token) for token in tokens])

#### Porter

In [ ]:
porter = PorterStemmer()

In [ ]:
data['porter_text'] = data['pre_text'].apply(lambda tokens: [porter.stem(token) for token in tokens])

### Using Lemmatizer

In [ ]:
lemm = WordNetLemmatizer()

In [ ]:
data['lemm_text'] = data['pre_text'].apply(lambda tokens: [lemm.lemmatize(token) for token in tokens])

In [ ]:
data.info()

# Spliting the data into Training and Test

In [ ]:
X = data.iloc[:,2:]

In [ ]:
y = data.iloc[:,0]

In [ ]:
print(X.shape, y.shape)

## Using only the preprocessed text

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X['pre_text'], y, test_size=0.2, random_state=42)

## Using the preprocessed text + snow stemmer

In [ ]:
X_train_snow, X_test_snow, y_train_snow, y_test_snow = train_test_split(X['snow_text'], y, test_size=0.2, random_state=42)

## Using the preprocessed text + porter stemmer

In [ ]:
X_train_porter, X_test_porter, y_train_porter, y_test_porter = train_test_split(X['porter_text'], y, test_size=0.2, random_state=42)

## Using the preprocessed text + noise removal

In [ ]:
X_train_filt, X_test_filt, y_train_filt, y_test_filt = train_test_split(X['pre_text_filter'], y, test_size=0.2, random_state=42)

## Using the preprocessed text + lemmatizer

In [ ]:
X_train_lemm, X_test_lemm, y_train_lemm, y_test_lemm = train_test_split(X['lemm_text'], y, test_size=0.2, random_state=42)

Defining a plotting function for Evaluation phase

In [ ]:
from sklearn.model_selection import learning_curve

def plot_learning_curve(
    model,
    X,
    y,
    scoring="accuracy",
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 5),
    title=None
):
    train_sizes, train_scores, val_scores = learning_curve(
        model,
        X,
        y,
        scoring=scoring,
        cv=cv,
        train_sizes=train_sizes,
        n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)

    plt.figure()
    plt.plot(train_sizes, train_mean, marker="o", label="Training score")
    plt.plot(train_sizes, val_mean, marker="o", label="Validation score")
    plt.xlabel("Training set size")
    plt.ylabel(scoring)
    plt.title(title or model.__class__.__name__)
    plt.legend()
    plt.grid(True)
    plt.show()

# Training some classifiers

### Only preprocessed text - Best(TF-IDF Passive Agressive Classifier - Acc: 91.27 %, Gini: 94.29 %)

#### TF-IDF - Best (Passive Agressive Classifier - Acc: 91.27 %, Gini: 94.29 %)

In [ ]:
vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_tfidf, y_train)
y_hat = dt_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_tfidf, y_train)
y_hat = log_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_tfidf, y_train)
y_hat = nb_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_tfidf, y_train)
y_hat = rf_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = np.argmax(rf_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1).fit(X_tfidf, y_train)
y_hat = pac.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = pac.decision_function(X_test_tfidf)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

#### BoW - Best (Naive Bayes - Acc: 93.01 %, Gini: 85.98 % )

In [ ]:
vectorizer = CountVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_bow, y_train)
y_hat = dt_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_bow, y_train)
y_hat = log_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_bow, y_train)
y_hat = nb_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_bow, y_train)
y_hat = rf_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = np.argmax(rf_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1).fit(X_bow, y_train)
y_hat = pac.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test, y_hat))

y_proba = pac.decision_function(X_test_bow)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

### Preprocessed text + noise removal - Best(TF-IDF Passive Agressive Classifier - Acc: 91.21 %, Gini: 93.88 %)

#### TF-IDF - Best (Passive Agressive Classifier - Acc: 91.21 %, Gini: 93.88 % )¶

In [ ]:
vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_tfidf = vectorizer.fit_transform(X_train_filt)
X_test_tfidf = vectorizer.transform(X_test_filt)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_tfidf, y_train_filt)
y_hat = dt_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_tfidf, y_train_filt)
y_hat = log_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_tfidf, y_train_filt)
y_hat = nb_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_tfidf, y_train_filt)
y_hat = rf_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = np.argmax(rf_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1).fit(X_tfidf, y_train_filt)
y_hat = pac.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = pac.decision_function(X_test_tfidf)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

#### BoW - Best (Passive Agressive Classifier - Acc: 90.95 %, Gini: 93.68 % )¶

In [ ]:
vectorizer = CountVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_bow = vectorizer.fit_transform(X_train_filt)
X_test_bow = vectorizer.transform(X_test_filt)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_bow, y_train_filt)
y_hat = dt_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_bow, y_train_filt)
y_hat = log_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_bow, y_train_filt)
y_hat = nb_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_bow, y_train_filt)
y_hat = rf_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = np.argmax(rf_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1).fit(X_bow, y_train_filt)
y_hat = pac.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_filt, y_hat))
print("Classification Report:\n", classification_report(y_test_filt, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_filt, y_hat))

y_proba = pac.decision_function(X_test_bow)
fpr, tpr, thresholds = roc_curve(y_test_filt, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

### Preprocessed text + snow stemmer - Best(TF-IDF Passive Agressive Classifier - Acc: 90.54 %, Gini: 93.15 %)

#### TF-IDF - Best (Passive Agressive Classifier - Acc: 90.54 %, Gini: 93.15 % )¶

In [ ]:
vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_tfidf = vectorizer.fit_transform(X_train_snow)
X_test_tfidf = vectorizer.transform(X_test_snow)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_tfidf, y_train_snow)
y_hat = dt_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_tfidf, y_train_snow)
y_hat = log_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_tfidf, y_train_snow)
y_hat = nb_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_tfidf, y_train_snow)
y_hat = rf_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = np.argmax(rf_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1).fit(X_tfidf, y_train_snow)
y_hat = pac.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = pac.decision_function(X_test_tfidf)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

#### BoW - Best (Passive Agressive Classifier - Acc: 90.26 %, Gini: 93.18 % )

In [ ]:
vectorizer = CountVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_bow = vectorizer.fit_transform(X_train_snow)
X_test_bow = vectorizer.transform(X_test_snow)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_bow, y_train_snow)
y_hat = dt_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_bow, y_train_snow)
y_hat = log_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_bow, y_train_snow)
y_hat = nb_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_bow, y_train_snow)
y_hat = rf_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=100, random_state=42, n_jobs=-1).fit(X_bow, y_train_snow)
y_hat = pac.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_snow, y_hat))
print("Classification Report:\n", classification_report(y_test_snow, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_snow, y_hat))

y_proba = pac.decision_function(X_test_bow)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

### Preprocessed text + porter stemmer - Best(TF-IDF Passive Agressive Classifier - Acc: 91.24 %, Gini: 93.65 %)

#### TF-IDF - Best (Passive Agressive Classifier - Acc: 91.24 %, Gini: 93.65 % )¶

In [ ]:
vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_tfidf = vectorizer.fit_transform(X_train_porter)
X_test_tfidf = vectorizer.transform(X_test_porter)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_tfidf, y_train_porter)
y_hat = dt_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_porter, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_tfidf, y_train_porter)
y_hat = log_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_porter, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_tfidf, y_train_porter)
y_hat = nb_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_porter, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_tfidf, y_train_porter)
y_hat = rf_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = np.argmax(rf_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_porter, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1).fit(X_tfidf, y_train_porter)
y_hat = pac.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = pac.decision_function(X_test_tfidf)
fpr, tpr, thresholds = roc_curve(y_test_porter, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

#### BoW - Best (Passive Agressive Classifier - Acc: 90.48 %, Gini: 93.17 % )¶

In [ ]:
vectorizer = CountVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_bow = vectorizer.fit_transform(X_train_porter)
X_test_bow = vectorizer.transform(X_test_porter)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_bow, y_train_porter)
y_hat = dt_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_bow, y_train_porter)
y_hat = log_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_bow, y_train_porter)
y_hat = nb_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_bow, y_train_porter)
y_hat = rf_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = np.argmax(rf_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_snow, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=100, random_state=42, n_jobs=-1).fit(X_bow, y_train_porter)
y_hat = pac.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_porter, y_hat))
print("Classification Report:\n", classification_report(y_test_porter, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_porter, y_hat))

y_proba = pac.decision_function(X_test_bow)
fpr, tpr, thresholds = roc_curve(y_test_porter, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

### Preprocessed text + lemmatizer - Best (Passive Agressive Classifier - Acc: 91.56 %, Gini: 94.03 %

#### TF-IDF - Best (Passive Agressive Classifier - Acc: 91.56 %, Gini: 94.03 % )¶

In [ ]:
vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_tfidf = vectorizer.fit_transform(X_train_lemm)
X_test_tfidf = vectorizer.transform(X_test_lemm)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_tfidf, y_train_lemm)
y_hat = dt_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_tfidf, y_train_lemm)
y_hat = log_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_tfidf, y_train_lemm)
y_hat = nb_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_tfidf, y_train_lemm)
y_hat = rf_classifier.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = np.argmax(rf_classifier.predict_proba(X_test_tfidf), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1).fit(X_tfidf, y_train_lemm)
y_hat = pac.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = pac.decision_function(X_test_tfidf)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

#### BoW - Best (Passive Agressive Classifier - Acc: 91.36 %, Gini: 93.7 % )¶

In [ ]:
vectorizer = CountVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False
)

X_bow = vectorizer.fit_transform(X_train_lemm)
X_test_bow = vectorizer.transform(X_test_lemm)

##### Decision Tree metrics

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42).fit(X_bow, y_train_lemm)
y_hat = dt_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = np.argmax(dt_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Logistic Regression metrics

In [ ]:
log_classifier = LogisticRegression(random_state=42).fit(X_bow, y_train_lemm)
y_hat = log_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = np.argmax(log_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Naive Bayes metrics

In [ ]:
nb_classifier = MultinomialNB().fit(X_bow, y_train_lemm)
y_hat = nb_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = np.argmax(nb_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### Random Forest metrics

In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_bow, y_train_lemm)
y_hat = rf_classifier.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = np.argmax(rf_classifier.predict_proba(X_test_bow), axis=1)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

##### PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1).fit(X_bow, y_train_lemm)
y_hat = pac.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = pac.decision_function(X_test_bow)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

## Bigrams

TF-IDF + Logistic Regression

Data Used => Processed Text + Lemmatizer

In [ ]:
vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False,
    # create word unigrams + bigrams
    ngram_range=(1, 2),
    # drops rare tokens/bigrams that appear once
    min_df=2,
    # drops near-constant boilerplate tokens
    max_df=0.9
)

X_tfidf = vectorizer.fit_transform(X_train_lemm)
X_test_tfidf = vectorizer.transform(X_test_lemm)

In [ ]:
log_reg = LogisticRegression(random_state=42).fit(X_tfidf, y_train_lemm)
y_hat = log_reg.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = log_reg.decision_function(X_test_tfidf)
fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print()
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

In [ ]:
plot_learning_curve(
    model = log_reg,
    X = X_tfidf,
    y = y_train_lemm,
    scoring="f1",
    title="Logistic Regression - TF-IDF (Unigrams + Bigrams)"
)

TF-IDF + PassiveAgressive Classifier

Data Used => Processed Text + Lemmatizer

In [ ]:
# .Setup the Vectorizer for Bigrams

tfidf_bigram = TfidfVectorizer(
    ngram_range=(1, 2),  # This is the key
    max_features=50000,   # Recommended: Bigrams explode feature count
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    # drops rare tokens/bigrams that appear once
    min_df=2,
    # drops near-constant boilerplate tokens
    max_df=0.9
)

X_train_tfidf = tfidf_bigram.fit_transform(X_train_lemm)
X_test_tfidf = tfidf_bigram.transform(X_test_lemm)

In [ ]:
# Fit the model
pac_bigram = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1)
pac_bigram.fit(X_train_tfidf, y_train_lemm)

In [ ]:
# 3. Predict
y_hat = pac_bigram.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_lemm, y_hat))
print("Classification Report:\n", classification_report(y_test_lemm, y_hat))
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_hat))

y_proba = pac_bigram.decision_function(X_test_tfidf)

fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print()
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

GridSearch PAC

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import classification_report

# 1. Define the Pipeline
# We use the lambda functions because your data is already lemmatized
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(tokenizer=lambda x: x, 
                              preprocessor=lambda x: x, 
                              token_pattern=None)),
    ('pac', PassiveAggressiveClassifier(random_state=42, n_jobs=-1))
])

# 2. Define the Parameters
# We test Unigrams vs Bigrams and the 'aggression' of the PAC (C)
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [30000, 50000],
    'pac__C': [0.1, 0.5, 1.0],
    'pac__max_iter': [50]
}

# 3. Run the GridSearch
# Using cv=3 to keep it fast; n_jobs=-1 uses all your CPU cores
grid = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=-1, scoring='accuracy', verbose=1)
grid.fit(X_train_lemm, y_train_lemm)

# 4. Results
print(f"Best Parameters: {grid.best_params_}")
print(f"Best Score: {grid.best_score_:.4f}")

# Final Evaluation
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_lemm)
print("\nFinal Classification Report:")
print(classification_report(y_test_lemm, y_pred))

# Confusion Matrix
print('Confusion matrix:\n', confusion_matrix(y_test_lemm, y_pred))

# Gini Coef.
y_proba = best_model.decision_function(X_test_lemm)

fpr, tpr, thresholds = roc_curve(y_test_lemm, y_proba)
print()
print('Gini coef.:', 2*(auc(fpr, tpr))-1)

Top 10 Weighted Features

In [ ]:
# 1. Access the trained pieces
best_model = grid.best_estimator_

tfidf = best_model.named_steps["tfidf"]
pac = best_model.named_steps["pac"]

# 2. Get feature names
feature_names = tfidf.get_feature_names_out()

# 3. Get coefficients
coefficients = pac.coef_[0]

# 4. Separate Fake vs Real features

# indices
top_fake_idx = np.argsort(coefficients)[-10:]
top_real_idx = np.argsort(coefficients)[:10]

top_fake = [(feature_names[i], coefficients[i]) for i in reversed(top_fake_idx)]
top_real = [(feature_names[i], coefficients[i]) for i in top_real_idx]

# 5. Print values
print("\nTop 10 Fake News Features:")
for word, weight in top_fake:
    print(f"{word:<25} {weight:.4f}")

print("\nTop 10 Real News Features:")
for word, weight in top_real:
    print(f"{word:<25} {weight:.4f}")


Plotting a Classification Report for best models

In [ ]:
# Data structure to hold the "Best of" results
best_results = [
    {"Model": "PAC (TF-IDF, Clean)", "Precision": 0.91, "Recall": 0.91, "F1": 0.91, "Accuracy": 0.9127, "Gini": 0.9428},
    {"Model": "LogReg (TF-IDF + Bigrams, Lemm)", "Precision": 0.93, "Recall": 0.93, "F1": 0.93, "Accuracy": 0.9276, "Gini": 0.9579},
    {"Model": "PAC (TF-IDF, Lemm)", "Precision": 0.92, "Recall": 0.92, "F1": 0.92, "Accuracy": 0.9156, "Gini": 0.9405},
    {"Model": "PAC (TF-IDF, Clean)", "Precision": 0.91, "Recall": 0.91, "F1": 0.91, "Accuracy": 0.9127, "Gini": 0.9429},
    {"Model": "PAC (TF-IDF + Bigrams, Lemm)", "Precision": 0.93, "Recall": 0.93, "F1": 0.93, "Accuracy": 0.929, "Gini": 0.9619},
    {"Model": "GridSearch PAC (TF-IDF + N-Gram + Dyn_C, Lemm)", "Precision": 0.94, "Recall": 0.94, "F1": 0.94, "Accuracy": 0.9290, "Gini": 0.9676}
]

df = pd.DataFrame(best_results)


In [ ]:

# sort by F1
df = df.sort_values("F1")

y = np.arange(len(df))
width = 0.4

plt.figure(figsize=(10, 6))

# bars
plt.barh(y - width/2, df["F1"], height=width, label="F1")
plt.barh(y + width/2, df["Accuracy"], height=width, label="Accuracy")

# gini as dots
plt.scatter(df["Gini"], y, label="Gini", zorder=3)

plt.yticks(y, df["Model"])
plt.xlabel("Score")
plt.title("Model Comparison: F1 vs Accuracy with Gini Overlay")
plt.xlim(0.9, 1.0)
plt.legend()
plt.tight_layout()
plt.show()